# BISINDO CNN+LSTM Training (Google Colab GPU)

Notebook ini menjalankan seluruh pipeline training di Colab dengan **import dataset dari Google Drive**:
1. Cek GPU
2. Clone repository & install dependencies
3. **Mount Google Drive** (tanpa upload manual)
4. Download dataset BISINDO alfabet (public GitHub repo)
5. Download klip gesture kata dari Google Drive via `gdown` + file ID
6. Preprocessing (MediaPipe Tasks API -> landmark -> sliding window -> `.npy`)
7. Augmentation + class balancing
8. Training CNN+LSTM dengan EarlyStopping & ReduceLROnPlateau
9. Evaluasi (classification report, confusion matrix)
10. Simpan model ke Google Drive + download `.h5` lokal

## 1. Cek GPU

In [ ]:
!nvidia-smi || echo 'GPU tidak terdeteksi - aktifkan di Runtime > Change runtime type'

## 2. Clone repository & install dependencies

In [ ]:
%cd /content
!rm -rf ML_Pak_Abdi
!git clone https://github.com/buble-max/ML_Pak_Abdi.git
%cd /content/ML_Pak_Abdi
!pip install -q -r requirements.txt

## 3. Mount Google Drive

Autentikasi Google Drive sekali supaya Colab dapat mengakses dataset/ model di `/content/drive/MyDrive/`. Ikuti instruksi popup (pilih akun Google, beri izin).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# (opsional) siapkan folder output di Drive
import os
DRIVE_PROJECT = '/content/drive/MyDrive/ML_Pak_Abdi'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print('Drive project dir:', DRIVE_PROJECT)

## 4. Download dataset alfabet BISINDO (public GitHub repo)

Dataset utama berada di folder `collectedimages/` repo GitHub `rhiosutoyo/...` - diunduh otomatis oleh script (bukan via ZIP upload).

In [ ]:
!python -m dataset.download_dataset

## 5. Download dataset tambahan dari Google Drive

Pengguna men-share **satu file ZIP** di Google Drive yang berisi struktur `raw_words/<LABEL>/clip_*/frame_*.jpg` (atau struktur `dataset/...` langsung). Pastikan file di-share **Anyone with the link**, lalu tempel URL (atau file ID) di sel berikut. Tidak perlu upload manual melalui browser.

Helper `dataset.download_from_drive` akan:
- Mengunduh ZIP via `gdown` (support file ID + URL `drive.google.com/file/d/...`).
- Membuat folder `dataset/` bila belum ada.
- Mengekstrak ZIP ke `dataset/`.
- Menghapus file ZIP setelah ekstraksi.
- Memvalidasi bahwa folder `raw_words/` berhasil dibuat.

In [ ]:
# Ganti dengan URL atau file ID dataset gesture kata Anda di Google Drive.
# Contoh URL: 'https://drive.google.com/file/d/1abcDEFghiJKLmnoPQRstuVWxyz/view?usp=sharing'
WORD_GESTURES_DRIVE_URL = 'PASTE_LINK_GOOGLE_DRIVE_DI_SINI'

from dataset.download_from_drive import download_and_extract

download_and_extract(
    file_id_or_url=WORD_GESTURES_DRIVE_URL,
    extract_to='dataset',
    expected_subdirs=['raw_words'],
    remove_zip=True,
)

!ls dataset/raw_words | head

### 5b. (Opsional) Import dataset live `.npy` dari Drive

Jika Anda merekam dataset live lewat `dataset/record_landmarks_live.py` di komputer lokal lalu upload ke Drive, import juga di sini. Folder target: `dataset/processed/live/`.

In [ ]:
# Kosongkan (None) jika tidak punya dataset live.
LIVE_DATASET_DRIVE_URL = None  # mis. 'https://drive.google.com/file/d/<FILE_ID>/view'

if LIVE_DATASET_DRIVE_URL:
    from dataset.download_from_drive import download_and_extract
    download_and_extract(
        file_id_or_url=LIVE_DATASET_DRIVE_URL,
        extract_to='dataset/processed/live',
        expected_subdirs=None,
        remove_zip=True,
    )
    !ls dataset/processed/live

## 6. Preprocessing -> `X.npy`, `y.npy`

In [ ]:
!python -m preprocessing.landmark_extractor

## 7. Augmentation + class balancing -> `X_aug.npy`, `y_aug.npy`

In [ ]:
!python -m augmentation.augment

## 8. Training CNN+LSTM

In [ ]:
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
!python -m training.train --epochs 100 --batch 64

## 9. Lihat hasil evaluasi

In [ ]:
from IPython.display import Image, display
print(open('logs/classification_report.txt').read())
display(Image('logs/confusion_matrix.png'))

In [ ]:
import json
import matplotlib.pyplot as plt

h = json.load(open('logs/history.json'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['loss'], label='train');      ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Loss');      ax[0].legend(); ax[0].grid(True)
ax[1].plot(h['accuracy'], label='train');  ax[1].plot(h['val_accuracy'], label='val')
ax[1].set_title('Accuracy');  ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 10. Simpan model ke Google Drive + download lokal

In [ ]:
import shutil, os

DRIVE_MODEL_DIR = os.path.join(DRIVE_PROJECT, 'saved_models')
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

shutil.copy2('model/saved/bisindo_model.h5', os.path.join(DRIVE_MODEL_DIR, 'bisindo_model.h5'))
shutil.copy2('model/saved/labels.json',      os.path.join(DRIVE_MODEL_DIR, 'labels.json'))
print('Model tersimpan di Drive:', DRIVE_MODEL_DIR)

In [ ]:
from google.colab import files
files.download('model/saved/bisindo_model.h5')
files.download('model/saved/labels.json')